In [18]:
from fishsense_api_sdk.client import Client
from fishsense_headtail_dataset.config import settings
from tqdm.asyncio import tqdm as tqdm_asyncio
from httpx import HTTPStatusError
from synology_api.filestation import FileStation
from urllib.parse import urlparse
from pathlib import Path
from tqdm.notebook import tqdm as tqdm
import cv2
from fishsense_core.image.raw_image import RawImage
from fishsense_core.image.rectified_image import RectifiedImage

In [19]:
DATA_FOLDER = (Path("./data") / "REEF" / "data").absolute()
OUTPUT_FOLDER = (Path("./output") / "headtail_dataset").absolute()

DATA_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

IMAGE_OUTPUT_FOLDER = OUTPUT_FOLDER / "images"
LABEL_OUTPUT_FOLDER = OUTPUT_FOLDER / "labels"

IMAGE_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
LABEL_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

DATA_FOLDER.exists(), OUTPUT_FOLDER.exists()

(True, True)

In [20]:
async with Client(settings.fishsense_api.url, settings.fishsense_api.username, settings.fishsense_api.password) as client:
    dives = await client.dives.get_canonical()

len(dives), dives

(272,
 [Dive(id=1, name='080123_FSL-01 Photos', path='2023-09-07 REEF Data Dump/080123_FSL-01 Photos', dive_datetime=datetime.datetime(2023, 8, 1, 12, 46, 27, tzinfo=datetime.timezone.utc), priority=<Priority.LOW: 'LOW'>, flip_dive_slate=True, camera_id=1, dive_slate_id=1),
  Dive(id=5, name='Hogfish01_MolHITW_0926_080323', path='2023.08.03.FishSense.FSL-01D/Hogfish01_MolHITW_0926_080323', dive_datetime=datetime.datetime(2023, 8, 3, 9, 27, 23, tzinfo=datetime.timezone.utc), priority=<Priority.LOW: 'LOW'>, flip_dive_slate=None, camera_id=1, dive_slate_id=None),
  Dive(id=8, name='DogSnapper01_MolPeLe_1024_080323', path='2023.08.03.FishSense.FSL-01D/DogSnapper01_MolPeLe_1024_080323', dive_datetime=datetime.datetime(2023, 8, 3, 10, 24, 59, tzinfo=datetime.timezone.utc), priority=<Priority.LOW: 'LOW'>, flip_dive_slate=None, camera_id=1, dive_slate_id=None),
  Dive(id=14, name='Hogfish01_ConchLed_1419_080323', path='2023.08.03.FishSense.FSL-01D/Hogfish01_ConchLed_1419_080323', dive_datetime

In [21]:
dives_by_id = {dive.id: dive for dive in dives}

len(dives_by_id), dives_by_id

(272,
 {1: Dive(id=1, name='080123_FSL-01 Photos', path='2023-09-07 REEF Data Dump/080123_FSL-01 Photos', dive_datetime=datetime.datetime(2023, 8, 1, 12, 46, 27, tzinfo=datetime.timezone.utc), priority=<Priority.LOW: 'LOW'>, flip_dive_slate=True, camera_id=1, dive_slate_id=1),
  5: Dive(id=5, name='Hogfish01_MolHITW_0926_080323', path='2023.08.03.FishSense.FSL-01D/Hogfish01_MolHITW_0926_080323', dive_datetime=datetime.datetime(2023, 8, 3, 9, 27, 23, tzinfo=datetime.timezone.utc), priority=<Priority.LOW: 'LOW'>, flip_dive_slate=None, camera_id=1, dive_slate_id=None),
  8: Dive(id=8, name='DogSnapper01_MolPeLe_1024_080323', path='2023.08.03.FishSense.FSL-01D/DogSnapper01_MolPeLe_1024_080323', dive_datetime=datetime.datetime(2023, 8, 3, 10, 24, 59, tzinfo=datetime.timezone.utc), priority=<Priority.LOW: 'LOW'>, flip_dive_slate=None, camera_id=1, dive_slate_id=None),
  14: Dive(id=14, name='Hogfish01_ConchLed_1419_080323', path='2023.08.03.FishSense.FSL-01D/Hogfish01_ConchLed_1419_080323', 

In [22]:
async def get_headtail_labels(client, dive):
    try:
        return await client.labels.get_headtail_labels(dive.id)
    except HTTPStatusError as e:
        if e.response.status_code == 404:
            return None
        raise

In [23]:
async with Client(settings.fishsense_api.url, settings.fishsense_api.username, settings.fishsense_api.password) as client:
    label_lists = await tqdm_asyncio.gather(*[get_headtail_labels(client, dive) for dive in dives])

label_lists = [label_list for label_list in label_lists if label_list is not None]
labels = [label for label_list in label_lists for label in label_list if label is not None]
len(labels), labels

100%|██████████| 272/272 [00:18<00:00, 14.63it/s]


(35680,
 [HeadTailLabel(id=2233, label_studio_task_id=157687, label_studio_project_id=45, head_x=None, head_y=None, tail_x=None, tail_y=None, updated_at=datetime.datetime(2025, 10, 23, 15, 38, 13, 23890, tzinfo=datetime.timezone.utc), superseded=False, completed=True, label_studio_json={'allow_skip': True, 'annotations': [{'id': 119089, 'result': [], 'created_username': 'Harrison Gaunt eci4@reef.org, 240', 'created_ago': '3\xa0months, 2\xa0weeks', 'completed_by': 240, 'was_cancelled': False, 'ground_truth': False, 'created_at': '2025-10-23T15:38:12.886643Z', 'updated_at': '2025-10-23T15:38:12.886654Z', 'draft_created_at': None, 'lead_time': 5.21, 'import_id': None, 'last_action': None, 'bulk_created': False, 'task': 157687, 'project': 45, 'updated_by': 240, 'parent_prediction': None, 'parent_annotation': None, 'last_created_by': None}], 'annotations_ids': '119089', 'annotations_results': '[]', 'annotators': [240], 'avg_lead_time': 5.21, 'cancelled_annotations': 0, 'comment_authors': []

In [24]:
async with Client(settings.fishsense_api.url, settings.fishsense_api.username, settings.fishsense_api.password) as client:
    images = await tqdm_asyncio.gather(*[client.images.get(image_id=label.image_id) for label in labels])

len(images), images

100%|██████████| 35680/35680 [00:49<00:00, 720.00it/s]


(35680,
 [Image(id=1, path='2023-09-07 REEF Data Dump/080123_FSL-01 Photos/P8010001.ORF', taken_datetime=datetime.datetime(2023, 8, 1, 11, 36, 47, tzinfo=datetime.timezone.utc), checksum='3055955122b63e02c8d7f9fcfb428989', is_canonical=True, dive_id=1, camera_id=1),
  Image(id=2, path='2023-09-07 REEF Data Dump/080123_FSL-01 Photos/P8010002.ORF', taken_datetime=datetime.datetime(2023, 8, 1, 11, 36, 47, tzinfo=datetime.timezone.utc), checksum='880ab67fbe636a88537a640f0cd22810', is_canonical=True, dive_id=1, camera_id=1),
  Image(id=3, path='2023-09-07 REEF Data Dump/080123_FSL-01 Photos/P8010003.ORF', taken_datetime=datetime.datetime(2023, 8, 1, 11, 36, 49, tzinfo=datetime.timezone.utc), checksum='45e90bab7a3897386ec27d4b82202f5f', is_canonical=True, dive_id=1, camera_id=1),
  Image(id=4, path='2023-09-07 REEF Data Dump/080123_FSL-01 Photos/P8010004.ORF', taken_datetime=datetime.datetime(2023, 8, 1, 11, 36, 49, tzinfo=datetime.timezone.utc), checksum='6c42938e52188b1566a9099a7b93a966', 

In [25]:
async with Client(settings.fishsense_api.url, settings.fishsense_api.username, settings.fishsense_api.password) as client:
    cameras = await tqdm_asyncio.gather(*[client.cameras.get(dive.camera_id) for dive in dives])

len(cameras), cameras

100%|██████████| 272/272 [00:00<00:00, 650.72it/s]


(272,
 [Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  Camera

In [26]:
cameras_by_id = {camera.id: camera for camera in cameras}

len(cameras_by_id), cameras_by_id

(7,
 {1: Camera(id=1, serial_number='BJ6C69506', name='FSL-01'),
  6: Camera(id=6, serial_number='BJ6C67987', name='FSL-06'),
  5: Camera(id=5, serial_number='BJ6C67988', name='FSL-05'),
  10: Camera(id=10, serial_number='BJ6C67989', name='FSL-07'),
  2: Camera(id=2, serial_number='BJ6C83748', name='FSL-02'),
  4: Camera(id=4, serial_number='BJ6C85524', name='FSL-04'),
  3: Camera(id=3, serial_number='BJ6C85528', name='FSL-03')})

In [27]:
async with Client(settings.fishsense_api.url, settings.fishsense_api.username, settings.fishsense_api.password) as client:
    camera_intrinsics_list = await tqdm_asyncio.gather(*[client.cameras.get_intrinsics(camera.id) for camera in cameras])

len(camera_intrinsics_list), camera_intrinsics_list

100%|██████████| 272/272 [00:00<00:00, 608.66it/s]


(272,
  <fishsense_api_sdk.models.camera_intrinsics.CameraIntrinsics at 0x713e534becf0>])

In [28]:
camera_intrinsics_by_camera_id = {intrinsics.camera_id: intrinsics for intrinsics in camera_intrinsics_list}

len(camera_intrinsics_by_camera_id), camera_intrinsics_by_camera_id

(7,
 {1: <fishsense_api_sdk.models.camera_intrinsics.CameraIntrinsics at 0x713e534becf0>,
  6: <fishsense_api_sdk.models.camera_intrinsics.CameraIntrinsics at 0x713e67544590>,
  5: <fishsense_api_sdk.models.camera_intrinsics.CameraIntrinsics at 0x713e534bc280>,
  10: <fishsense_api_sdk.models.camera_intrinsics.CameraIntrinsics at 0x713ee370ad60>,
  2: <fishsense_api_sdk.models.camera_intrinsics.CameraIntrinsics at 0x713e67546580>,
  4: <fishsense_api_sdk.models.camera_intrinsics.CameraIntrinsics at 0x713e534a9b70>,
  3: <fishsense_api_sdk.models.camera_intrinsics.CameraIntrinsics at 0x713e5347e430>})

In [29]:
parsed_url = urlparse(settings.e4e_nas.url)

filestation = FileStation(parsed_url.hostname, parsed_url.port, settings.e4e_nas.username, settings.e4e_nas.password, secure=True, cert_verify=False)

In [30]:
for label, image in tqdm(zip(labels, images), total=len(labels)):
    dive = dives_by_id[image.dive_id]
    camera = cameras_by_id[dive.camera_id]
    camera_intrinsics = camera_intrinsics_by_camera_id[camera.id]

    image_path = DATA_FOLDER / image.path
    image_target_path = IMAGE_OUTPUT_FOLDER / f"{image.checksum}.JPG"
    label_target_path = LABEL_OUTPUT_FOLDER / f"{image.checksum}.txt"

    source_nas_path = f"/fishsense_data/REEF/data/{image.path}"
    filestation.get_file(source_nas_path, "download", dest_path=str(image_path.parent))

    image = RectifiedImage(RawImage(image_path), camera_intrinsics)
    img = image.data

    cv2.imwrite(image_target_path.as_posix(), img)

    if label.head_x is None or label.head_y is None or label.tail_x is None or label.tail_y is None:
        continue
    
    with open(label_target_path, "w") as f:
        f.write(f"{label.head_x} {label.head_y} {label.tail_x} {label.tail_y}\n")

  0%|          | 0/35680 [00:00<?, ?it/s]

KeyboardInterrupt: 